# 🧠 AI Logic Engine: High-Precision Knowledge Ingestion Notebook

This notebook is designed to generate high-quality knowledge triplets at scale using a free LLM (Qwen) and ingest them into your Firestore Symbolic Database. It supports interrupted runs and automatic resumption.

### 🚀 Key Features:
- **Neural-Free Parsing**: Uses high-precision symbolic regex matching.
- **Scalable Ingestion**: Batched Firestore writes for efficiency.
- **Resume Capability**: Tracks progress to avoid duplicate work.
- **Free Hardware**: Optimized for Google Colab Free T4 GPU.

---

In [ ]:
# @title 🛠️ Step 1: Install Dependencies & Setup Environment
!pip install -q transformers accelerate bitsandbytes firebase-admin tqdm

import os
import json
import re
import torch
from tqdm.auto import tqdm
import firebase_admin
from firebase_admin import credentials, firestore
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from google.colab import drive

print("✅ Setup complete. GPU detected: ", torch.cuda.is_available())

In [ ]:
# @title 🔑 Step 2: Firebase Connection Setup
# @markdown 1. Go to [Firebase Console](https://console.firebase.google.com/project/gen-lang-client-0802662324/settings/serviceaccounts/adminsdk)
# @markdown 2. Click **"Generate new private key"** and save the JSON file.
# @markdown 3. Run this cell to upload that file directly to Colab.

from google.colab import files
import firebase_admin
from firebase_admin import credentials, firestore

print("Please upload your serviceAccountKey.json file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2' # @param {type:"string"}

if not firebase_admin._apps:
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Connected to Firestore Database: {DATABASE_ID}")

In [ ]:
# @title 🤖 Step 3: Load Free High-Quality Model (Qwen2.5)
model_id = "Qwen/Qwen2.5-1.5B-Instruct" # Lightweight but very precise for facts

print("⏳ Loading model... this might take 2-3 minutes.")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)
print("✅ Model loaded into VRAM.")

In [ ]:
# @title 🧬 Step 4: Symbolic Parser Port (Python Version)

def parse_text_to_triplets(text):
    triplets = []
    # Split by common markers
    sentences = re.split(r'[.!?၊။\n\r]+', text)
    
    for s in sentences:
        s = s.strip()
        if not s or len(s) < 2: continue
        
        # Clean ending particles
        clean_s = re.sub(r'[ပါတော်မူ၏လဲဗျာရှင်နော်ဦးအုံးဟယ်]+$', '', s).strip()
        
        # Pattern 1: Multi-Subject with 'and'
        multi_sub = re.match(r'^(.*?)(?:နှင့်|နှင့်|‌ေရာ)\s+(.*?)(?:သည်|က|မှ|၏)\s+(.*?)(?:ကို|အား|ထံ)\s+(.*?)(?:သည်|၏|ခဲ့သည်|နေသည်|ပါသည်)$', clean_s)
        if multi_sub:
            triplets.append({'s': multi_sub.group(1).strip(), 'v': multi_sub.group(4).strip(), 'o': multi_sub.group(3).strip()})
            triplets.append({'s': multi_sub.group(2).strip(), 'v': multi_sub.group(4).strip(), 'o': multi_sub.group(3).strip()})
            continue
            
        # Pattern 2: SOV Particle Based (Subject ... Object ... Verb)
        sov = re.match(r'^(.*?)(?:သည်|က|မှ|၏)\s+(.*?)(?:ကို|အား|ထံ)\s+(.*?)(?:သည်|၏|ခဲ့သည်|နေသည်|ပါသည်)$', clean_s)
        if sov:
            triplets.append({'s': sov.group(1).strip(), 'v': sov.group(3).strip(), 'o': sov.group(2).strip()})
            continue
            
        # Pattern 3: Simple Predicate "is a / is property"
        pred = re.match(r'^(.*?)(?:သည်|က|၏)\s+(.*?)(?:ဖြစ်သည်|ဖြစ်ပါသည်|သည်|၏)$', clean_s)
        if pred:
            triplets.append({'s': pred.group(1).strip(), 'v': 'is_a', 'o': pred.group(2).strip()})
            continue
            
        # Pattern 4: Location "is at"
        loc = re.match(r'^(.*?)\s+(.*?)(?:မှာ|မှ|ထံ|ဆီ)(?:သို့|က)?\s+(.*?)(?:သည်|၏|ခဲ့သည်|နေသည်|ပါသည်)$', clean_s)
        if loc:
            triplets.append({'s': loc.group(1).strip(), 'v': 'is_at', 'o': loc.group(2).strip()})
            continue
            
    return triplets

print("✅ Symbolic Parser Ready.")

In [ ]:
# @title 📦 Step 5: High-Scale Ingestion Engine (Batch Tracking)

def add_to_firestore(s, v, o):
    """Port of LogicEngine.addTriplet logic for Python Firestore Admin"""
    s_key = s.lower().strip()
    o_key = o.lower().strip() if o else 'null'
    
    # Note: For speed in 1M scale, we use batched writes
    # We'll manage the batch externally to this function
    return s_key, o_key

def generate_batch(prompt, count=35):
    messages = [
        {"role": "system", "content": "You are a specializes Knowledge Graph Generator. Generate strictly factual, diverse, and unique sentences."},
        {"role": "user", "content": f"Generate {count} unique, specific factual sentences in Burmese or English. One per line."}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(model_inputs.input_ids, max_new_tokens=1536, do_sample=True, temperature=0.7)
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.split("assistant\\n")[-1]

print("✅ Ingestion Engine Configured.")

In [ ]:
# @title 🚀 Step 6: RUN ALL - Start Ingesting (1M Capacity)

SOURCE_MODE = "AI Generated" # @param ["AI Generated", "Uploaded File"]
TARGET_GOAL = 1000000 # @param {type:"integer"}
USE_DRIVE_STORAGE = True # @param {type:"boolean"}
BATCH_SAVE_INTERVAL = 500

import queue
import threading
from google.colab import drive

# Interactive override if needed
try:
    custom_goal = input(f"Target Goal is set to {TARGET_GOAL}. Enter a new number to override or press Enter to continue: ")
    if custom_goal.strip():
        TARGET_GOAL = int(custom_goal)
        print(f"🎯 Goal updated to: {TARGET_GOAL}")
except Exception as e:
    print(f"Using default goal: {TARGET_GOAL}")

PROGRESS_FILE = 'ingestion_progress.json'
if USE_DRIVE_STORAGE:
    print("📂 Mounting Google Drive for persistent progress tracking...")
    drive.mount('/content/drive', force_remount=True)
    PROGRESS_FILE = '/content/drive/MyDrive/logic_engine_progress.json'

# Load Progress
current_count = 0
if os.path.exists(PROGRESS_FILE):
    try:
        with open(PROGRESS_FILE, 'r') as f:
            data = json.load(f)
            current_count = data.get('count', 0)
            print(f"🔄 Resuming from {current_count} facts (Last saved state).")
    except: print("⚠️ Progress file corrupted, starting from 0.")

pbar = tqdm(total=TARGET_GOAL, initial=current_count, desc="Ingestion Progress")
write_queue = queue.Queue(maxsize=200)
stop_event = threading.Event()

def firestore_worker():
    """Background worker for high-speed firestore writes"""
    global current_count
    while not stop_event.is_set() or not write_queue.empty():
        try:
            triplets = write_queue.get(timeout=2)
            batch = db.batch()
            for t in triplets:
                # 1. Subject ID Normalization (Match Server Logic)
                s_id = t['s'].strip()
                s_key = s_id.lower()[:128]
                if not s_key: continue
                
                doc_ref = db.collection('nodes').document(s_key)
                batch.set(doc_ref, {
                    'id': s_id,
                    'relations': firestore.ArrayUnion([{'verb': t['v'], 'targetId': t['o'], 'weight': 100}]),
                    'groups': firestore.ArrayUnion([t['o']]) if t['v'] == 'is_a' else firestore.ArrayUnion([]),
                    'type': 'ENTITY',
                    'updatedAt': firestore.SERVER_TIMESTAMP
                }, merge=True)
            
            batch.commit()
            current_count += len(triplets)
            pbar.update(len(triplets))
            
            # Periodic Save
            if current_count % BATCH_SAVE_INTERVAL == 0:
                with open(PROGRESS_FILE, 'w') as f:
                    json.dump({'count': current_count}, f)
                    
        except queue.Empty: continue
        except Exception as e:
            if "Quota exceeded" in str(e):
                print("\n🛑 FIREBASE QUOTA EXCEEDED. Pausing.")
                stop_event.set()
            else:
                print(f"\n⚠️ Write Error: {e}")

writer_thread = threading.Thread(target=firestore_worker)
writer_thread.start()

try:
    if SOURCE_MODE == "AI Generated":
        print(f"🚀 Generating Knowledge (Goal: {TARGET_GOAL})...")
        while current_count < TARGET_GOAL and not stop_event.is_set():
            # Use specialized prompt for high quality facts
            raw_output = generate_batch("Generate detailed, non-trivial facts about science, history, linguistics, and geography.", count=20)
            triplets = parse_text_to_triplets(raw_output)
            if triplets:
                write_queue.put(triplets)
    else:
        print("📂 Please upload your .txt file (Sentences per line):")
        uploaded = files.upload()
        if uploaded:
            file_path = list(uploaded.keys())[0]
            with open(file_path, 'r') as f:
                lines_batch = []
                for line in f:
                    triplets = parse_text_to_triplets(line)
                    lines_batch.extend(triplets)
                    if len(lines_batch) >= 50:
                        write_queue.put(lines_batch)
                        lines_batch = []
                if lines_batch: write_queue.put(lines_batch)

except KeyboardInterrupt:
    print("\n🛑 Paused by user. Cleaning up...")
finally:
    stop_event.set()
    writer_thread.join()
    pbar.close()
    with open(PROGRESS_FILE, 'w') as f:
        json.dump({'count': current_count}, f)
    print(f"✅ Session Ended. Final Fact Count: {current_count}")